# 043 数据清洗：用户信息数据

In [17]:
USER_INFO_PATH = r'..\data\fe\user_info.parquet'

TOPIC_COMMENT_PATH = r"..\data\cleaned\topic_comment.parquet"
USER_WEIBO_PATH = r"..\data\cleaned\user_weibo.parquet"

In [18]:
import pandas as pd

df_user_info = pd.read_parquet(USER_INFO_PATH)

df_topic_comment = pd.read_parquet(TOPIC_COMMENT_PATH)
df_user_weibo = pd.read_parquet(USER_WEIBO_PATH)


In [19]:
# 记录表原始行数，用于全程追踪保留率
_ORIGINAL_COUNTS = {
    "user_info": len(df_user_info),
}

def retention_rate(table: str, current_df) -> str:
    """计算相对于原始数据的保留率。"""
    orig = _ORIGINAL_COUNTS[table]
    cur = len(current_df)
    return f"{cur:,} / {orig:,} ({cur / orig * 100:.1f}% 保留)"

print("\n📦 原始数据量:")
print(f"  user_info:     {len(df_user_info):>10,}")


📦 原始数据量:
  user_info:         11,012


In [20]:
import re

def clean_text(text: str, type: str) -> str:
    """清洗微博评论文本，保留情绪信号。

    清洗步骤（有序）：
    1. 移除 HTML 标签
    2. 移除 URL
    3. 移除 回复@xxx: 前缀（保留正文部分）
    4. 规范化空白字符
    """
    if not isinstance(text, str) or len(text) == 0:
        return text

    # 1. 移除 HTML 标签
    text = re.sub(r'<[^>]+>', '', text)

    # 2. 移除 URL
    text = re.sub(r'https?://\S+', '', text)

    # 3. 移除 "回复@xxx：" 或 "回复@xxx:" 前缀，保留后续正文
    if type == "comment":
        text = re.sub(r'^回复@[\w\u4e00-\u9fff]+[：:]', '', text)

    # 4. 话题标签：#xxx# → xxx（保留标签文字内容）
    # text = re.sub(r'#([^#]+)#', r'\1', text)

    # 5. 规范化空白（多个空白合并为一个，去首尾空白）
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [21]:
# ========== 用户活跃度和内容质量统计 ==========
print("\n\n📊 开始用户活跃度和内容质量统计...")

# 5.1 微博统计
print("\n📝 微博统计...")
weibo_stats = df_user_weibo.groupby("user_id").agg(
    weibo_crawled_count=("weibo_id", "nunique"),
    weibo_hq_count=("text_quality", lambda x: (x >= 3).sum()),
    original_ratio=("is_repost", lambda x: 1 - x.mean()),

).reset_index()
weibo_stats["weibo_hq_ratio"] = (weibo_stats["weibo_hq_count"] / weibo_stats["weibo_crawled_count"]).round(2)

print(f"✅ 微博统计完成：{len(weibo_stats):,} 个用户")

# 5.2 评论统计
print("\n💬 评论统计...")
comment_stats = df_topic_comment.groupby("user_id").agg(
    comment_crawled_count=("comment_id", "count"),
    comment_hq_count=("text_quality", lambda x: (x >= 3).sum()),
).reset_index()
comment_stats["comment_hq_ratio"] = (comment_stats["comment_hq_count"] / comment_stats["comment_crawled_count"]).round(2)

print(f"✅ 评论统计完成：{len(comment_stats):,} 个用户")

# 5.3 活跃天数统计
print("\n📅 活跃天数统计...")
# 从微博中统计活跃天数
weibo_active_days = df_user_weibo.groupby("user_id")["create_time"].apply(
    lambda x: x.dt.date.nunique()
).reset_index()
weibo_active_days.columns = ["user_id", "weibo_active_days"]

# 从评论中统计活跃天数
comment_active_days = df_topic_comment.groupby("user_id")["create_time"].apply(
    lambda x: x.dt.date.nunique()
).reset_index()
comment_active_days.columns = ["user_id", "comment_active_days"]

# 合并两个活跃天数，取最大值（用户在微博或评论中任一天活跃即计为活跃）
active_days = weibo_active_days.merge(comment_active_days, on="user_id", how="outer").fillna(0).astype({
    "weibo_active_days": int,
    "comment_active_days": int
})
active_days["active_days"] = active_days[["weibo_active_days", "comment_active_days"]].max(axis=1)
active_days = active_days[["user_id", "active_days"]]

print(f"✅ 活跃天数统计完成：{len(active_days):,} 个用户")

# 5.4 将统计结果合并到 df_user_info
df_user_info = df_user_info.merge(weibo_stats, on="user_id", how="left")
df_user_info = df_user_info.merge(comment_stats, on="user_id", how="left")
df_user_info = df_user_info.merge(active_days, on="user_id", how="left")

# 5.5 填充缺失值
df_user_info["weibo_crawled_count"] = df_user_info["weibo_crawled_count"].fillna(0).astype(int)
df_user_info["weibo_hq_count"] = df_user_info["weibo_hq_count"].fillna(0).astype(int)
df_user_info["weibo_hq_ratio"] = df_user_info["weibo_hq_ratio"].fillna(0.0).round(2)
df_user_info["original_ratio"] = df_user_info["original_ratio"].fillna(0).round(2)

df_user_info["comment_crawled_count"] = df_user_info["comment_crawled_count"].fillna(0).astype(int)
df_user_info["comment_hq_count"] = df_user_info["comment_hq_count"].fillna(0).astype(int)
df_user_info["comment_hq_ratio"] = df_user_info["comment_hq_ratio"].fillna(0.0).round(2)
df_user_info["active_days"] = df_user_info["active_days"].fillna(0).astype(int)

print(f"\n✅ 用户统计已回填到 df_user_info")
print(f"\n📈 统计概览:")
print(f"  微博爬取数: {df_user_info['weibo_crawled_count'].describe()}")
print(f"  评论爬取数: {df_user_info['comment_crawled_count'].describe()}")
print(f"  活跃天数: {df_user_info['active_days'].describe()}")



📊 开始用户活跃度和内容质量统计...

📝 微博统计...
✅ 微博统计完成：46,008 个用户

💬 评论统计...
✅ 微博统计完成：46,008 个用户

💬 评论统计...
✅ 评论统计完成：83,606 个用户

📅 活跃天数统计...
✅ 评论统计完成：83,606 个用户

📅 活跃天数统计...
✅ 活跃天数统计完成：116,700 个用户

✅ 用户统计已回填到 df_user_info

📈 统计概览:
  微博爬取数: count    11012.000000
mean        38.863422
std         27.105345
min          0.000000
25%         32.000000
50%         44.000000
75%         50.000000
max       1354.000000
Name: weibo_crawled_count, dtype: float64
  评论爬取数: count    11012.000000
mean         3.084363
std          5.058271
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max        158.000000
Name: comment_crawled_count, dtype: float64
  活跃天数: count    11012.000000
mean        16.618053
std         13.727380
min          1.000000
25%          6.000000
50%         14.000000
75%         25.000000
max        351.000000
Name: active_days, dtype: float64
✅ 活跃天数统计完成：116,700 个用户

✅ 用户统计已回填到 df_user_info

📈 统计概览:
  微博爬取数: count    11012.000000
mean        38.86342

In [22]:
# ========== 5.4 description 清洗 ==========
df_user_info["description"] = df_user_info["description"].apply(
    lambda x: clean_text(x, type="description") if isinstance(x, str) else x
).replace('', None)

In [23]:
# df_user_info 最终字段
user_info_cols = [
    # ID
    "user_id", "screen_name", "gender",
    # 位置
    "ip_location",
    # 账号信息
    "registration_time", "account_age_days",
    "verified", "verified_type", "verified_type_name",
    # 社交指标
    "total_weibo_count", "follower_count", "following_count",
    "follower_following_ratio", "user_rank",
    # 微博内容统计
    "weibo_crawled_count", "weibo_hq_count", "weibo_hq_ratio", "original_ratio",
    # 评论内容统计
    "comment_crawled_count", "comment_hq_count", "comment_hq_ratio",
    # 活跃度
    "active_days",
    # 个人简介
    "description",
]
df_user_info = df_user_info[user_info_cols]

In [39]:
# ========== 用户价值分类 ==========
print("\n\n📊 开始用户价值分类...")

def classify_user_value(row):
    """
    根据用户特征分类用户价值等级。
    
    返回值: (user_value, user_value_label)
    """
    # 1. 非普通个体用户判断
    if row['verified_type_name'] not in ['个人认证', '普通用户']:
        return -1, "非普通个体用户"
    
    # 2. 低可用用户判断
    if (row['weibo_hq_count'] < 7 or 
        row['weibo_crawled_count'] < 20 or 
        row['active_days'] < 6):
        return 0, "低可用用户"
    
    # 3. 核心建模候选用户判断（最严格条件）
    if (row['weibo_crawled_count'] >= 44 and 
        row['weibo_hq_count'] >= 31 and 
        row['weibo_hq_ratio'] >= 0.84 and 
        row['original_ratio'] >= 0.44 and 
        row['active_days'] >= 14 and 
        row['comment_hq_count'] >= 2):
        return 3, "核心建模候选用户"
    
    # 4. 高可用用户判断
    if (row['weibo_crawled_count'] >= 32 and 
        row['weibo_hq_count'] >= 31 and 
        row['weibo_hq_ratio'] >= 0.84 and 
        row['active_days'] >= 14):
        return 2, "高可用用户"
    
    # 5. 其余为一般可用用户
    return 1, "一般可用用户"

# 应用分类逻辑
df_user_info[['user_value', 'user_value_label']] = df_user_info.apply(
    lambda row: pd.Series(classify_user_value(row)), axis=1
)

print(f"✅ 用户价值分类完成")
print(f"\n📈 用户价值等级分布:")
value_counts = df_user_info['user_value_label'].value_counts()
for label, count in value_counts.items():
    pct = count / len(df_user_info) * 100
    print(f"  {label:>20}: {count:>6,} ({pct:>5.1f}%)")




📊 开始用户价值分类...
✅ 用户价值分类完成

📈 用户价值等级分布:
                 低可用用户:  4,109 ( 37.3%)
                一般可用用户:  4,060 ( 36.9%)
                 高可用用户:  1,882 ( 17.1%)
              核心建模候选用户:    809 (  7.3%)
               非普通个体用户:    152 (  1.4%)


In [40]:
import os

# 创建输出目录
output_dir = r"..\data\cleaned"
os.makedirs(output_dir, exist_ok=True)

# 保存
datasets = {
    "user_info": df_user_info,
}

for name, df in datasets.items():
    path = os.path.join(output_dir, f"{name}.parquet")
    df.to_parquet(path, index=False)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"✅ {name}.parquet 已保存 ({df.shape[0]:>10,} rows × {df.shape[1]:>2} cols, {size_mb:.1f} MB)")

print(f"\n📂 输出目录: {os.path.abspath(output_dir)}")

✅ user_info.parquet 已保存 (    11,012 rows × 25 cols, 0.9 MB)

📂 输出目录: d:\GraduationProject\data\cleaned
